# Lab 10 — SVD: Geometry and Image Compression
**Linear Algebra · UATX**

Every matrix $A \in \mathbb{R}^{m \times n}$ factors as $A = U\Sigma V^\top$ where $U, V$ are orthogonal and $\Sigma = \mathrm{diag}(\sigma_1 \geq \cdots \geq \sigma_r > 0)$. The **Eckart–Young theorem** says $A_k = \sum_{i=1}^k \sigma_i u_i v_i^\top$ is the best rank-$k$ approximation in the Frobenius norm.

**Tasks**
1. Visualise the SVD of a $2 \times 2$ matrix as mapping the unit circle to an ellipse.
2. Plot Frobenius reconstruction error vs rank $k$ for a random matrix.
3. Apply SVD compression to real images; plot singular-value histograms and energy deciles.
4. *(New)* Find the minimum rank $k^*$ for 95% energy retention; report the compression ratio.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import imageio.v3 as iio
np.random.seed(42)

## §1  SVD Geometry: Unit Circle → Ellipse

The SVD decomposes $A = U\Sigma V^\top$. Applied to the unit circle:
$$A\cdot(\cos\theta, \sin\theta)^\top = U\Sigma(V^\top\text{circle})$$
The right singular vectors $v_i$ are the input axes; the images $\sigma_i u_i$ are the ellipse semiaxes.

In [ ]:
A = np.array([[3., 0.], [1., 2.]])
U, s, Vt = np.linalg.svd(A)

theta = np.linspace(0, 2*np.pi, 300)
circle = np.vstack([np.cos(theta), np.sin(theta)])
ellipse = A @ circle

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(circle[0], circle[1], 'b-', label='unit circle', lw=1.5, alpha=0.6)
ax.plot(ellipse[0], ellipse[1], 'r-', label='image $A\\cdot$circle', lw=2)

# Draw input singular vectors v_i
for i in range(2):
    v = Vt.T[:, i]
    ax.annotate('', xy=v, xytext=(0,0),
                arrowprops=dict(arrowstyle='->', color='steelblue', lw=2))
    ax.text(v[0]*1.1, v[1]*1.1, f'$v_{i+1}$', fontsize=12, color='steelblue')

# Draw output singular vectors sigma_i * u_i
for i in range(2):
    sv = s[i] * U[:, i]
    ax.annotate('', xy=sv, xytext=(0,0),
                arrowprops=dict(arrowstyle='->', color='tomato', lw=2))
    ax.text(sv[0]*1.05, sv[1]*1.05, f'$\\sigma_{i+1}u_{i+1}$', fontsize=11, color='tomato')

ax.set_aspect('equal'); ax.grid(True); ax.legend()
ax.set_title(f'SVD of $A$: singular values $\\sigma_1={s[0]:.2f}$, $\\sigma_2={s[1]:.2f}$')
plt.tight_layout(); plt.show()

print(f'U:\n{np.round(U,4)}')
print(f'Sigma: {np.round(s,4)}')
print(f'Vt:\n{np.round(Vt,4)}')
print(f'\nA = U Sigma Vt?  {np.allclose(A, U @ np.diag(s) @ Vt)}')

## §2  Low-Rank Approximation: Frobenius Error vs Rank $k$

The Eckart–Young theorem: $\|A - A_k\|_F = \sqrt{\sigma_{k+1}^2 + \cdots + \sigma_r^2}$. The error decays as we include more singular values.

In [ ]:
np.random.seed(0)
M = np.random.randn(50, 50)
U_M, s_M, Vt_M = np.linalg.svd(M, full_matrices=False)

k_vals = list(range(1, 51))
errors = [np.linalg.norm(M - (U_M[:,:k] * s_M[:k]) @ Vt_M[:k,:], 'fro') for k in k_vals]

# Theoretical: tail of singular values
errors_theory = [np.sqrt(np.sum(s_M[k:]**2)) for k in k_vals]

plt.figure(figsize=(8, 4))
plt.plot(k_vals, errors, 'b-o', ms=4, lw=2, label='actual $\\|M - M_k\\|_F$')
plt.plot(k_vals, errors_theory, 'r--', lw=2, label='theory $\\sqrt{\\sum_{i>k}\\sigma_i^2}$')
plt.xlabel('Rank $k$'); plt.ylabel('Frobenius error')
plt.title('Eckart–Young: low-rank approximation error')
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

print('Max discrepancy (theory vs actual):', np.max(np.abs(np.array(errors) - errors_theory)))

## §3  Image Compression via SVD

An image is a matrix. We compress it to rank $k$, storing $k(m+n)$ values instead of $mn$.

In [ ]:
image_names = ['astronaut', 'chelsea', 'coffee', 'camera']
images = {}
for name in image_names:
    img = iio.imread(f'imageio:{name}.png').astype(float) / 255.0
    if img.ndim == 3: img = img.mean(axis=2)
    images[name] = img[:600, :600]

ranks = [5, 20, 60]

for name, img in images.items():
    U_i, s_i, Vt_i = np.linalg.svd(img, full_matrices=False)

    # Energy deciles
    total_energy = np.sum(s_i**2)
    cumulative   = np.cumsum(s_i**2) / total_energy
    decile_pcts  = np.arange(10, 110, 10)
    decile_svs   = np.percentile(s_i, [p for p in range(10, 100, 10)])

    fig, axes = plt.subplots(1, len(ranks)+2, figsize=(16, 3))
    fig.suptitle(name.capitalize())

    axes[0].imshow(img, cmap='gray'); axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].semilogy(s_i); axes[1].set_title('Singular values')
    axes[1].set_xlabel('index'); axes[1].grid(True)

    for ax, k in zip(axes[2:], ranks):
        Mk = (U_i[:,:k] * s_i[:k]) @ Vt_i[:k,:]
        err = np.linalg.norm(img - Mk, 'fro')
        ax.imshow(np.clip(Mk, 0, 1), cmap='gray')
        ax.set_title(f'Rank {k}\n$\\|\\cdot\\|_F$={err:.1f}'); ax.axis('off')

    plt.tight_layout(); plt.show()

## §4  95% Energy Rank and Compression Ratio  *(New)*

For each image, find the smallest $k^*$ such that
$$\frac{\sum_{i=1}^{k^*}\sigma_i^2}{\sum_i \sigma_i^2} \geq 0.95.$$
The **compression ratio** is $k^*(m+n)/(mn)$: values below 1 mean the compressed form takes fewer numbers.

In [ ]:
print(f'{"Image":<12} {"m x n":<12} {"k* (95%)":<12} {"full size":<12} {"compressed":<14} {"ratio"}')
print('-'*68)
for name, img in images.items():
    m, n = img.shape
    _, s_i, _ = np.linalg.svd(img, full_matrices=False)
    total_energy  = np.sum(s_i**2)
    cumulative    = np.cumsum(s_i**2) / total_energy
    k_star = int(np.searchsorted(cumulative, 0.95)) + 1
    ratio  = k_star * (m + n) / (m * n)
    print(f'{name.capitalize():<12} {m}x{n:<8} {k_star:<12} {m*n:<12} {k_star*(m+n):<14} {ratio:.3f}')

print('\n(ratio < 1 means compression saves storage)')

In [ ]:
# Visualise cumulative energy curves
fig, ax = plt.subplots(figsize=(8, 4))
for name, img in images.items():
    _, s_i, _ = np.linalg.svd(img, full_matrices=False)
    cumulative = np.cumsum(s_i**2) / np.sum(s_i**2)
    ax.plot(cumulative, label=name.capitalize())
ax.axhline(0.95, color='red', ls='--', label='95% energy')
ax.set_xlabel('Rank $k$'); ax.set_ylabel('Fraction of total energy $\\sum_{i\\leq k}\\sigma_i^2 / \\sum \\sigma_i^2$')
ax.set_title('Cumulative energy vs. rank'); ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()

In [ ]:
# Verify: A^T A = V Sigma^2 V^T (eigendecomposition connection)
img_test = images['camera']
U_t, s_t, Vt_t = np.linalg.svd(img_test, full_matrices=False)
AtA = img_test.T @ img_test
AtA_svd = Vt_t.T @ np.diag(s_t**2) @ Vt_t
print('A^T A = V Sigma^2 V^T?', np.allclose(AtA, AtA_svd, atol=1e-5))

# Eigenvalues of A^T A = squared singular values of A
eigs_AtA = np.linalg.eigvalsh(AtA)
print('Eigenvalues of A^T A match sigma^2?',
      np.allclose(np.sort(eigs_AtA)[-len(s_t):], np.sort(s_t**2), rtol=1e-4))